In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

/Users/jillchow/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
#clone the git repo that contains the data and additional information about the dataset
# !git clone https://github.com/wayfair/WANDS.git

Cloning into 'WANDS'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 45 (delta 10), reused 20 (delta 3), pack-reused 0
Receiving objects: 100% (45/45), 33.33 MiB | 30.69 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [9]:
#define functions for product search using Tf-IDF
def calculate_tfidf(dataframe):
    """
    Calculate the TF-IDF for combined product name and description.

    Parameters:
    dataframe (pd.DataFrame): DataFrame with product_id, and other product information.

    Returns:
    TfidfVectorizer, csr_matrix: TF-IDF vectorizer and TF-IDF matrix.
    """
    # Combine product name and description to vectorize
    # NOTE: Please feel free to use any combination of columns available, some columns may contain NULL values
    combined_text = dataframe['product_name'] + ' ' + dataframe['product_description']
    vectorizer = TfidfVectorizer()
    # convert combined_text to list of unicode strings
    tfidf_matrix = vectorizer.fit_transform(combined_text.values.astype('U'))
    return vectorizer, tfidf_matrix

def get_top_products(vectorizer, tfidf_matrix, query, top_n=10):
    """
    Get top N products for a given query based on TF-IDF similarity.

    Parameters:
    vectorizer (TfidfVectorizer): Trained TF-IDF vectorizer.
    tfidf_matrix (csr_matrix): TF-IDF matrix for the products.
    query (str): Search query.
    top_n (int): Number of top products to return.

    Returns:
    list: List of top N product IDs.
    """
    query_vector = vectorizer.transform([query])
    cosine_similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_product_indices = cosine_similarities.argsort()[-top_n:][::-1]
    return top_product_indices

In [10]:
#define functions for evaluating retrieval performance
def map_at_k(true_ids, predicted_ids, k=10):
    """
    Calculate the Mean Average Precision at K (MAP@K).

    Parameters:
    true_ids (list): List of relevant product IDs.
    predicted_ids (list): List of predicted product IDs.
    k (int): Number of top elements to consider.
             NOTE: IF you wish to change top k, please provide a justification for choosing the new value

    Returns:
    float: MAP@K score.
    """
    #if either list is empty, return 0
    if not len(true_ids) or not len(predicted_ids):
        return 0.0

    score = 0.0
    num_hits = 0.0

    for i, p_id in enumerate(predicted_ids[:k]):
        if p_id in true_ids and p_id not in predicted_ids[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(len(true_ids), k)

In [138]:
# Please add any new evaluation functions here

In [2]:
# get search queries
# query_df = pd.read_csv("WANDS/dataset/query.csv", sep='\t')


# load the data 
# 1. config project path

import sys
import os
import json 
import pandas as pd
import numpy as np 

ENV = 'dev'

## TODO: set the production environment in docker 

# Load config
with open("config.json") as f:
    config = json.load(f)

if ENV == 'dev':
    base_path = config[f"{ENV}_path"]  
    data_path = os.path.join(base_path, "data")
    model_path = os.path.join(base_path, "models")
    print("Base path:", base_path)    
    print("Data path:", data_path)
    print("Model path:", model_path)


# query_df = pd.read_csv()

Base path: /Users/jillchow/HBS/hbs_search_engine
Data path: /Users/jillchow/HBS/hbs_search_engine/data
Model path: /Users/jillchow/HBS/hbs_search_engine/models


In [ ]:
queryfile_name = "query.csv" 
queryfile_path = os.path.join(data_path, queryfile_name)
productfile_name = "product.csv" 
productfile_path = os.path.join(data_path, productfile_name)
labelfile_name = "label.csv" 
labelfile_path = os.path.join(data_path, labelfile_name)


query_df = pd.read_csv(queryfile_path, sep='\t')
product_df = pd.read_csv(productfile_path, sep='\t')
label_df = pd.read_csv(labelfile_path, sep='\t')

print('query_df: search queries')
display(query_df.head()) # watch for null values in query class column 
query_df.info()

print('\n product_df: product information')
display(product_df.head())
product_df.info() # watch for null values other than product_id, product_name, product_features

print('\n label_df: ground truth labels')
display(label_df.head())
label_df.info()

query_df: search queries


,query_id,query,query_class
0,0,salon chair,Massage Chairs
1,1,smart coffee table,Coffee & Cocktail Tables
2,2,dinosaur,Kids Wall Décor
3,3,turquoise pillows,Accent Pillows
4,4,chair and a half recliner,Recliners


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   query_id     480 non-null    int64 
 1   query        480 non-null    object
 2   query_class  474 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.4+ KB

 product_df: product information


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42994 entries, 0 to 42993
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           42994 non-null  int64  
 1   product_name         42994 non-null  object 
 2   product_class        40142 non-null  object 
 3   category hierarchy   41438 non-null  object 
 4   product_description  36986 non-null  object 
 5   product_features     42994 non-null  object 
 6   rating_count         33542 non-null  float64
 7   average_rating       33542 non-null  float64
 8   review_count         33542 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 3.0+ MB

 label_df: ground truth labels


,id,query_id,product_id,label
0,0,0,25434,Exact
1,1,0,12088,Irrelevant
2,2,0,42931,Exact
3,3,0,2636,Exact
4,4,0,42923,Exact


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233448 entries, 0 to 233447
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          233448 non-null  int64 
 1   query_id    233448 non-null  int64 
 2   product_id  233448 non-null  int64 
 3   label       233448 non-null  object
dtypes: int64(3), object(1)
memory usage: 7.1+ MB


In [38]:
#group the labels for each query to use when identifying exact matches
grouped_label_df = label_df.groupby('query_id')
print(label_df.query_id.nunique())
print(product_df.product_id.nunique(), product_df.shape[0])

480
42994 42994


In [11]:
# Calculate TF-IDF
vectorizer, tfidf_matrix = calculate_tfidf(product_df)

In [ ]:
print(product_df.shape, tfidf_matrix.shape)
# this generates the tfidf_matrix for all products in the dataset, 
# with shape (number of products, number of features), 

query = "armchair"
query_vector = vectorizer.transform([query]) 
print(query_vector.shape)

# the cosine similairty does matrix multiplication between the query vector and the tfidf_matrix
cosine_similarity(query_vector, tfidf_matrix).shape

# with shape (1, number of features) * (number of features, number of products) = (1, number of products)
# then flattne the result to get a 1D array of cosine similarity scores


(42994, 9) (42994, 44307)
(1, 44307)


(1, 42994)

In [ ]:
query = "armchair"
query_vector = vectorizer.transform([query]) 
query_vector.shape
cosine_similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
print(cosine_similarities.shape)
top_product_indices = cosine_similarities.argsort()[-10:][::-1]
print(type(top_product_indices))

print(f"Top products for '{query}':")
for product_id in product_df.iloc[top_product_indices]['product_id']:
    product = product_df.loc[product_df['product_id'] == product_id]
    print(product_id, product['product_name'].values[0])

(42994,)
<class 'numpy.ndarray'>
Top products for 'armchair':
12756 24.41 '' wide tufted polyester armchair
42698 donham armchair
42697 donham 25 '' wide armchair
41270 almaraz 33.7 '' wide leather match armchair
23907 faizah 27.6 '' wide tufted polyester armchair
31564 biloxi 34.75 '' wide armchair
41306 hartsell 33 '' wide armchair
1527 howington 39 '' wide tufted linen armchair
42802 donham polyester lounge chair
6532 ogan 29 '' wide polyester armchair


In [26]:
display(product_df[product_df['product_id']==12756])

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
12756,12756,24.41 '' wide tufted polyester armchair,Accent Chairs,Furniture / Living Room Furniture / Chairs & S...,nothing makes a contemporary design statement ...,backheight-seattotopofback:14|levelofassembly ...,NaN,NaN,NaN


In [30]:
#Sanity check code block to see if the search results are relevant
#implementing a function to retrieve top K product IDs for a query
def get_top_product_ids_for_query(query):
    top_product_indices = get_top_products(vectorizer, tfidf_matrix, query, top_n=10)
    top_product_ids = product_df.iloc[top_product_indices]['product_id'].tolist()
    return top_product_ids

#define the test query
query = "armchair"

#obtain top product IDs
top_product_ids = get_top_product_ids_for_query(query)

print(f"Top products for '{query}':")
for product_id in top_product_ids:
    product = product_df.loc[product_df['product_id'] == product_id]
    print(product_id, product['product_name'].values[0])

Top products for 'armchair':
12756 24.41 '' wide tufted polyester armchair
42698 donham armchair
42697 donham 25 '' wide armchair
41270 almaraz 33.7 '' wide leather match armchair
23907 faizah 27.6 '' wide tufted polyester armchair
31564 biloxi 34.75 '' wide armchair
41306 hartsell 33 '' wide armchair
1527 howington 39 '' wide tufted linen armchair
42802 donham polyester lounge chair
6532 ogan 29 '' wide polyester armchair


In [ ]:
#implementing a function to retrieve exact match product IDs for a query_id
# def get_exact_matches_for_query(query_id):
#     query_group = grouped_label_df.get_group(query_id)
#     exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
#     return exact_matches

# query_df['top_product_ids'] = query_df['query'].apply(get_top_product_ids_for_query)
# query_df.head()

# query_df['relevant_ids'] = query_df['query_id'].apply(get_exact_matches_for_query)
# query_df.head()

query_group = grouped_label_df.get_group(0)
exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
# print(f"Exact matches for query_id 0: {exact_matches}")
# print(type(exact_matches))
# query_group = grouped_label_df.get_group(query_id)
# exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
# return exact_matches


Exact matches for query_id 0: [25434 42931  2636 42923 41156  5936 22390 42929 42928 25428 39428 39429
 25433 25432 25431 27534 36910 25429  2638 27541 29746 29744 20026 22391
  7465 25427 25426  1197 15612  7468 25424 25423 39461  1198  7467 42927
  7466 25425 22389]
<class 'numpy.ndarray'>


In [41]:
#implementing a function to retrieve exact match product IDs for a query_id
def get_exact_matches_for_query(query_id):
    query_group = grouped_label_df.get_group(query_id)
    exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
    return exact_matches

#applying the function to obtain top product IDs and adding top K product IDs to the dataframe 
query_df['top_product_ids'] = query_df['query'].apply(get_top_product_ids_for_query)

#adding the list of exact match product_IDs from labels_df
query_df['relevant_ids'] = query_df['query_id'].apply(get_exact_matches_for_query)

#now assign the map@k score
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)


In [46]:
# calculate the MAP across the entire query set
query_df.loc[:, 'map@k'].mean()

0.29319550540123457

In [51]:
# query_df.head(10)

def clean_query(query):
    """
    Clean the query string by removing special characters and converting to lowercase.

    Parameters:
    query (str): The input query string.

    Returns:
    str: Cleaned query string.
    """
    # Remove special characters and convert to lowercase
    cleaned_query = ''.join(e for e in query if e.isalnum() or e.isspace()).lower()
    return cleaned_query

query_df['cleaned_query'] = query_df['query'].apply(clean_query)
query_df[['query', 'cleaned_query']].head(10)

def count_words(clean_query:str):
    """
    Count the number of words in a cleaned query string.

    Parameters:
    clean_query (str): The cleaned query string.

    Returns:
    int: Number of words in the cleaned query.
    """
    # Count the number of words
    word_count = len(clean_query.split())
    return word_count


query_df['word_count'] = query_df['cleaned_query'].apply(count_words)



In [53]:
query_df['word_count'].describe()

count    480.000000
mean       3.377083
std        1.486718
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max       10.000000
Name: word_count, dtype: float64